# Fine-tune the skin scanner model on a GPU

This notebook trains the scanner's model with every layer learning, which the laptop cannot do, and gives you back one file to install.

**Before you run anything**

1. Menu **Runtime > Change runtime type**, choose **T4 GPU**, and save.
2. Have `colab_pack.zip` ready. It is in the project's `build` folder (create it with `.venv\Scripts\python.exe colab\make_pack.py`).

**Then**

3. Menu **Runtime > Run all**. Step 2 asks you to pick `colab_pack.zip`.
4. Keep this browser tab open. The whole run takes about one hour.
5. At the end your browser downloads `colab_result.zip`. The last section says what to do with it.

## Step 1: check that the GPU is on

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, save, and run again."
print("GPU:", torch.cuda.get_device_name(0))

## Step 2: upload `colab_pack.zip`

A file picker appears below the cell. Choose `colab_pack.zip` from the project's `build` folder.

In [ ]:
import os
import zipfile

from google.colab import files

os.chdir("/content")
if not os.path.exists("/content/colab_pack.zip"):
    uploaded = files.upload()
    name = next((n for n in uploaded if n.lower().endswith(".zip")), None)
    assert name, "No zip file was uploaded. Run this cell again and choose colab_pack.zip."
    os.replace(name, "/content/colab_pack.zip")
os.makedirs("/content/skin", exist_ok=True)
with zipfile.ZipFile("/content/colab_pack.zip") as pack:
    pack.extractall("/content/skin")
os.chdir("/content/skin")
assert os.path.exists("build/unified_manifest.json"), "This zip is not a colab_pack.zip made by colab/make_pack.py."
print("Project unpacked in /content/skin")

## Step 3: install the two extra packages

In [ ]:
%cd /content/skin
!pip -q install "openvino==2025.4.1" "timm==1.0.31"
import cv2, openvino, scipy, sklearn, timm, torch

print("torch", torch.__version__, "| timm", timm.__version__, "| openvino", openvino.__version__, "| opencv", cv2.__version__)

## Step 4: download the photos and check them

This downloads the five public Kaggle datasets (about 9.6 GB), keeps the 35,189 photos the project uses, and checks every one against the project's list. It takes about 10 minutes.

*Only if this step stops and says Kaggle wants a login:* on kaggle.com open **Settings > API > Create New Token**, set `NEED_KAGGLE_LOGIN = True` in the next cell, run it, pick the `kaggle.json` you got, and then run step 4 again.

In [ ]:
NEED_KAGGLE_LOGIN = False  # change to True only if the download below asked for it

if NEED_KAGGLE_LOGIN:
    import os

    from google.colab import files

    os.chdir("/content/skin")
    token = files.upload()
    assert "kaggle.json" in token, "Please upload the file named kaggle.json"
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.replace("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("Kaggle token saved")

In [ ]:
%cd /content/skin
!python colab/setup_data.py
assert _exit_code == 0, "The photos could not be prepared. Read the message above."

## Step 5: train

About 45 minutes on a T4. One line is printed per epoch; `val composite` is the score the project uses to pick the best epoch.

If the tab loses its connection but the runtime is still alive, run this cell again: it continues from the last finished epoch.

In [ ]:
%cd /content/skin
!python backend/finetune_gpu.py --resume --out build/colab_result
assert _exit_code == 0, "Training stopped with an error. Read the message above."

## Step 6: download the result

In [ ]:
from google.colab import files

files.download("/content/skin/build/colab_result/colab_result.zip")

## Back on the laptop

Put `colab_result.zip` in the project folder (`C:\xampp\htdocs\IPT_ABANCIO`) and run this in PowerShell from that folder, with the charger plugged in:

```powershell
$env:SKIN_TRAIN_THREADS = '6'
.\.venv\Scripts\python.exe backend\install_bundle.py colab_result.zip
```

It scores the new model on the laptop exactly as the scanner runs it (about 15 minutes), shows it next to the installed model, and installs it only if it is better. The replaced model is backed up in `build\model_backups`.

### Optional: a larger model

`--model base` in step 5 trains DINOv2 ViT-B/14 instead of ViT-S/14. It takes about three times as long to train, the model file is about four times larger, and each scan takes about four times longer on the laptop. Try the default first.